# vLLM check on Colab

Runs the batched vLLM runner (`run_vllm.py`) on problems that already have Transformers results, so the two backends can be compared before the main grid uses vLLM:

- the 10-pair arithmetic pilot at 4k tokens, compared with `results/pilot_10_pairs/` in the repo;
- the length-2 calibration pair at 4k, 8k, and 16k tokens, compared with the `budget_check` results in Drive if that notebook has finished.

vLLM samples differently from Transformers, so individual answers will differ. Accuracy, token use, and how often each system hits its cap should be similar. The notebook also checks that every problem stayed within its budget and reports generation speed.

Use **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/vllm_pilot/` and `results/vllm_budget_check/`, each with a `run.log`. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/small-multi-agents/results"
EXPERIMENTS = {
    "vllm_pilot": {"problems": "results/pilot_10_pairs/problems.jsonl", "budgets": [4000]},
    "vllm_budget_check": {"problems": "results/chain_calibration/d2.jsonl", "budgets": [4000, 8000, 16000]},
}
for name in EXPERIMENTS:
    !mkdir -p {RESULTS}/{name}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner scripts run as subprocesses, so no runtime restart is needed. The versions in use are saved to `environment.txt` in each results folder.

In [ ]:
!pip install -q -r requirements-vllm.txt
for name in EXPERIMENTS:
    !python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {RESULTS}/{name}/environment.txt
    !git rev-parse HEAD | tee -a {RESULTS}/{name}/environment.txt

## Run

Each experiment loads the model once and runs all three systems at each budget. Problems go to vLLM in batches, and each finished batch is saved before the next starts.

In [ ]:
from notebook_tools import run_logged

for name, spec in EXPERIMENTS.items():
    out = f"{RESULTS}/{name}"
    run_logged(["python", "run_vllm.py", spec["problems"], "--out-dir", out,
                "--budgets", *map(str, spec["budgets"]), "--seed", "42", "--resume"], f"{out}/run.log")

## Results

One row per problem for each backend. Transformers rows come from the repo pilot logs and, if present, from the `budget_check` results in Drive. `parts` lists each ensemble attempt's or hierarchy stage's status.

In [ ]:
import json
from pathlib import Path

import pandas as pd


def load_runs(root, backend):
    rows = []
    for path in sorted(Path(root).glob("b*/*.jsonl")):
        for line in path.read_text().splitlines():
            r = json.loads(line)
            if "attempts" in r:
                parts = [a["status"] for a in r["attempts"]]
            elif "calls" in r:
                parts = [f'{c["stage"]}:{c["stop_reason"]}' for c in r["calls"]]
            else:
                parts = []
            rows.append({
                "backend": backend, "system": path.stem, "budget": r["budget"], "pair_id": r["pair_id"],
                "kind": r["kind"], "status": r["status"], "parts": ", ".join(parts),
                "input": r["input_tokens"], "output": r["output_tokens"], "total": r["total_tokens"],
                "seconds": r.get("seconds"),
            })
    return rows


def load_pilot_hf():
    rows = []
    for system in ("single_agent", "ensemble", "hierarchy"):
        for line in Path(f"results/pilot_10_pairs/{system}.jsonl").read_text().splitlines():
            r = json.loads(line)
            rows.append({
                "backend": "transformers", "system": system, "budget": r["budget"], "pair_id": r["pair_id"],
                "kind": r["kind"], "status": r["status"], "parts": "", "input": r["input_tokens"],
                "output": r["output_tokens"], "total": r["total_tokens"], "seconds": r.get("seconds"),
            })
    return rows


pilot = pd.DataFrame(load_runs(f"{RESULTS}/vllm_pilot", "vllm") + load_pilot_hf())
budget = pd.DataFrame(
    load_runs(f"{RESULTS}/vllm_budget_check", "vllm") + load_runs(f"{RESULTS}/budget_check", "transformers")
)
pilot.to_csv(f"{RESULTS}/vllm_pilot/summary_problems.csv", index=False)
budget.to_csv(f"{RESULTS}/vllm_budget_check/summary_problems.csv", index=False)


def compare(df):
    return df.groupby(["system", "budget", "backend"]).agg(
        correct=("status", lambda s: (s == "correct").sum()),
        problems=("status", "size"),
        cap_hits=("parts", lambda p: p.str.contains("length_limit").sum()),
        mean_input=("input", "mean"),
        mean_output=("output", "mean"),
        mean_total=("total", "mean"),
    )


pilot_by_system = compare(pilot)
pilot_by_system.to_csv(f"{RESULTS}/vllm_pilot/summary_by_system.csv")
pilot_by_system

In [ ]:
budget_by_system = compare(budget)
budget_by_system.to_csv(f"{RESULTS}/vllm_budget_check/summary_by_system.csv")
pd.set_option("display.max_colwidth", None)
display(budget_by_system)
budget.sort_values(["budget", "system", "kind", "backend"])

## Checks

Every vLLM problem must stay within its budget. Generation speed is output tokens per second of batch wall time. Batched problems share wall time, so per-problem `seconds` are equal shares of their batch.

In [ ]:
vllm_rows = pd.concat([pilot[pilot.backend == "vllm"], budget[budget.backend == "vllm"]])
over = vllm_rows[vllm_rows.total > vllm_rows.budget]
print("problems over budget:", len(over))
speed = vllm_rows.groupby(["system", "budget"]).agg(output=("output", "sum"), seconds=("seconds", "sum"))
speed["output_tok_per_s"] = (speed.output / speed.seconds).round(1)
speed